## Task 1

- {종목명}의 {날짜} 종가는?
- {날짜} 거래량 기준 상위 10개의 종목은?

In [1]:
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
from pykrx import stock
import os
import dotenv
from langchain_naver import ChatClovaX
import re
from typing import Literal, List, Tuple

dotenv.load_dotenv()

clova = ChatClovaX(
    api_key=os.getenv('NCP_CLOVASTUDIO_API_KEY'),
    model="HCX-005",
    temperature=0.2,
    max_tokens=None,
    timeout=None,
    # max_retries,
    # repeat_penalt,
    # repetition_penalty,
    # stop,
    # seed,
    # top_k,
    # top_p,
)

# HCX-005
# 입력 토큰과 출력 토큰의 합은 128,000 토큰을 초과할 수 없습니다.
# 입력 토큰은 최대 128,000 토큰까지 가능합니다.
# 모델에 요청할 출력 토큰(maxTokens)은 최대 4,096 토큰까지 가능합니다.

In [2]:
# # def name_to_ticker(kor_name: str, date: str | None = None, csv_path="krx_tickers.csv"):
#     """코스피/코스닥 한글명 → 6자리 코드.KS/KQ"""
#     # date = date or datetime.today().strftime("%Y%m%d")
#     # tickers = stock.get_market_ticker_list(date)
#     # for t in tickers:
#     #     if stock.get_market_ticker_name(t) == kor_name:
#     #         return f"{t}.KS" if t[0] == "0" else f"{t}.KQ"
#     # return None

def name_to_ticker(kor_name: str, csv_path="krx_tickers.csv"):
    '''
    코스피/코스닥 한글명(종목명) → 6자리 코드.KS/KQ(티커명)
    '''

    df = pd.read_csv(csv_path)
    ticker = df[kor_name]
    return f"{ticker}.KS" if ticker[0] == "0" else f"{ticker}.KQ"

def normalize_date(date_str: str | None = None) -> str:
    """
    • date_str가 None 이면 → 오늘 날짜
    • '오늘', '어제', '그제', '일주일전' 같은 한국어 상대표현 처리
    • 숫자·슬래시·'월/일' 포맷 등 폭넓게 지원
    ▸ 반환: 'YYYY-MM-DD'
    """
    # 1) None → 오늘
    if date_str is None or str(date_str).strip() == "":
        return datetime.today().strftime("%Y-%m-%d")

    s = date_str.strip()

    # 2) 상대 표현 매핑
    today = datetime.today()
    rel_map = {
        r"^오늘$": 0,
        r"^어제$": -1,
        r"^그제$": -2,
        r"^(\d+)일[전 ago]*$": lambda m: -int(m.group(1)),
        r"^(일|1)주일?전$": -7,
        r"^지난주$": -7,
        r"^(한|1)달전$": -30,
    }
    for pat, delta in rel_map.items():
        m = re.fullmatch(pat, s, re.I)
        if m:
            offset = delta(m) if callable(delta) else delta
            return (today + timedelta(days=offset)).strftime("%Y-%m-%d")

    # 3) YYYYMMDD
    if re.fullmatch(r"\d{8}", s):
        return datetime.strptime(s, "%Y%m%d").strftime("%Y-%m-%d")

    # 4) ISO(YYYY-MM-DD)
    try:
        return datetime.strptime(s, "%Y-%m-%d").strftime("%Y-%m-%d")
    except ValueError:
        pass

    # 5) 'M월D일', 'MM/DD', 'MM.DD'
    m = re.fullmatch(r"(\d{1,2})\s*월\s*(\d{1,2})\s*일", s)
    if m:
        month, day = map(int, m.groups())
        return datetime(today.year, month, day).strftime("%Y-%m-%d")
    m = re.fullmatch(r"(\d{1,2})[./\s](\d{1,2})", s)
    if m:
        month, day = map(int, m.groups())
        return datetime(today.year, month, day).strftime("%Y-%m-%d")

    # 6) 파이널 폴백
    try:
        return pd.to_datetime(s).strftime("%Y-%m-%d")
    except Exception:
        raise ValueError(f"지원하지 않는 날짜 형식: {date_str}")
    
def get_price(stock_name: str, date: str, field: str = "close"):
    '''
    시가,고자,저가,종가 조회
    ex) 삼성전자의 5월28일 종가는?
    '''
    try:
        date = normalize_date(date)
    except ValueError as e:
        return f"❌ {e}"

    ticker = name_to_ticker(stock_name)
    if not ticker:
        return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."
    
    d0 = pd.Timestamp(date)
    d1 = d0 + pd.Timedelta(days=1)
    df = yf.download(ticker, start=d0.strftime("%Y-%m-%d"), end=d1.strftime("%Y-%m-%d"),
                     progress=False)
    if df.empty:
        return f"❌ {date} 거래 데이터가 없습니다."
    mapping = {"open": "Open", "high": "High", "low": "Low", "close": "Close"}
    price = df[mapping[field.lower()]].iloc[0]
    return f"{stock_name} {date} {field.upper()}는 {price:,.0f}원입니다."

def get_volume(stock_name: str, date: str | None = None):
    """
    고정된 날짜의 거래량 조회 (날짜 생략 시 오늘)
    예) "삼성전자 거래량?"  -> 오늘 기준
        "삼성전자 어제 거래량?" -> 전일 기준
    """
    try:
        date_std = normalize_date(date)
    except ValueError as e:
        return f"❌ {e}"

    ticker = name_to_ticker(stock_name, date_std.replace("-", ""))
    if not ticker:
        return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."

    d0 = pd.Timestamp(date_std)
    d1 = d0 + pd.Timedelta(days=1)
    df = yf.download(ticker, start=d0.strftime("%Y-%m-%d"),
                     end=d1.strftime("%Y-%m-%d"), progress=False)

    if df.empty:
        return f"❌ {date_std} 거래 데이터가 없습니다."

    vol = int(df["Volume"].iloc[0])
    return f"{stock_name}의 {date_std} 거래량은 {vol:,}주입니다."

def get_volume_range(stock_name: str, start_date: str, end_date: str, aggregate: str = "sum"):
    '''
    기간별 거래량 조회
    - aggregate="sum"   → 기간 총합
    - aggregate="daily" → 일자별 상세표
    '''
    try:
        start_std = normalize_date(start_date)
        end_std   = normalize_date(end_date)
    except ValueError as e:
        return f"❌ {e}"

    if start_std > end_std:
        return "❌ 시작 날짜가 종료 날짜보다 늦습니다."

    ticker = name_to_ticker(stock_name, start_std.replace("-", ""))
    if not ticker:
        return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."

    # yfinance 범위 다운로드 (end는 포함 안 됨 → 하루 더)
    df = yf.download(
        ticker,
        start=start_std,
        end=(pd.Timestamp(end_std) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        progress=False,
    )

    if df.empty:
        return f"❌ {start_std}~{end_std} 구간에 거래 데이터가 없습니다."

    vol_series = df["Volume"]

    if aggregate == "daily":
        lines = [f"{d.date()}: {v:,}주" for d, v in vol_series.items()]
        return f"{stock_name} {start_std}~{end_std} 일별 거래량:\n" + "\n".join(lines)

    # 기본: 총합
    total_vol = int(vol_series.sum())
    return f"{stock_name}의 {start_std}~{end_std} 기간 총 거래량은 {total_vol:,}주입니다."

def get_change_pct(stock_name: str, date_from: str, date_to: str | None = None):
    """
    기간 대비 주가 변동률 계산 함수
    ex) “삼성전자 5월 28일 전일 대비 상승률은?”
        “삼성전자 5/02에서 5/04까지 주가 변동률 알려줘”
        “삼성전자 지난주 대비 얼마 올랐어?” (LLM이 날짜 파싱해 전달)
        “삼전 전주 대비 등락폭?”
    """

    try:
        date_from = normalize_date(date_from)
        date_to  = normalize_date(date_to)
    except ValueError as e:
        return f"❌ {e}"

    # date_to가 None이면 '전일 대비' 로 처리
    if date_to is None:
        d0 = pd.Timestamp(date_from)
        # d1 = d0 - pd.Timedelta(days=7)  # 주말 보정
        date_to = (d0 - pd.tseries.offsets.BDay(1)).strftime("%Y-%m-%d")
    
    ticker = name_to_ticker(stock_name, date_from.replace("-", ""))
    if not ticker:
        return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."
    
    df = yf.download(ticker, start=date_to, end=pd.Timestamp(date_from)+pd.Timedelta(days=1),
                     progress=False)
    if len(df) < 2:
        return f"❌ 비교 구간 데이터가 부족합니다."
    
    price_from, price_to = df["Close"].iloc[[0, -1]]
    pct = (price_to / price_from - 1) * 100
    return f"{stock_name}의 {date_to} → {date_from} 변동률은 {pct:.2f}%입니다."

def _top_by(date: str, metric: Literal["volume", "gain", "drop"], n: int = 10,) -> List[Tuple[str, float]]:
    '''
    거래량‧상승률‧하락률 TOP N 종목 뽑는 함수
    ex) 5월28일에서 거래량 기준 상위 10개 종목은?
        5월 2일 하락률 1~3위는?
        오늘 상승률 상위 5개 종목?
    '''
    try:
        date_std = normalize_date(date)
    except ValueError as e:
        return f"❌ {e}"
    
    krx_codes = stock.get_market_ticker_list(date_std.replace("-", ""))
    if not krx_codes:
        raise ValueError(f"{date_std} 기준 상장 종목을 찾을 수 없습니다.")

    # 2) Yahoo 코드 (.KS/.KQ) 변환
    yf_codes = [f"{c}.KS" if c[0] == "0" else f"{c}.KQ" for c in krx_codes]

    def chunks(lst, size):  # 간단한 제네레이터
        for i in range(0, len(lst), size):
            yield lst[i : i + size]

    d0 = pd.Timestamp(date_std)
    d1 = d0 + pd.Timedelta(days=1)
    frames = []
    for batch in chunks(yf_codes, 200):
        try:
            df = yf.download(
                batch,
                start=d0.strftime("%Y-%m-%d"),
                end=d1.strftime("%Y-%m-%d"),
                group_by="ticker",
                progress=False,
            )
            frames.append(df)
        except Exception as e:
            print(f"[WARN] yfinance batch 실패: {e}")
    if not frames:
        raise RuntimeError("야후 파이낸스 데이터를 가져오지 못했습니다.")
    data = pd.concat(frames, axis=1)

    rows: List[Tuple[str, float]] = []
    for code in yf_codes:
        try:
            s = data[code]
            if s.empty:
                continue
            if metric == "volume":
                val = int(s["Volume"].iloc[0])
            else:
                open_, close_ = s["Open"].iloc[0], s["Close"].iloc[0]
                pct = (close_ / open_ - 1) * 100
                val = pct
            rows.append((code[:-3], val))  # ".KS" → 제거
        except KeyError:
            continue

    if not rows:
        raise RuntimeError("계산 가능한 종목이 없습니다.")

    # 5) 정렬 및 상위 N 추출
    reverse = metric != "drop"  # drop(하락률)은 오름차순
    rows.sort(key=lambda x: x[1], reverse=reverse)
    return rows[: n]

def get_top_volume(date: str, top_n: int = 10):
    rows = _top_by(date, "volume", n=top_n, reverse=True)
    txt = [f"{i+1}. {nm} - {vol:,}주" for i, (nm, vol) in enumerate(rows)]
    return f"{date} 거래량 TOP {top_n}:\n" + "\n".join(txt)

def get_top_gain(date: str, top_n: int = 10):
    rows = _top_by(date, "gain", n=top_n, reverse=True)
    txt = [f"{i+1}. {nm} ▲ {chg:.2f}%" for i, (nm, chg) in enumerate(rows)]
    return f"{date} 상승률 TOP {top_n}:\n" + "\n".join(txt)

def get_top_drop(date: str, top_n: int = 10):
    rows = _top_by(date, "drop", n=top_n, reverse=False)
    txt = [f"{i+1}. {nm} ▼ {chg:.2f}%" for i, (nm, chg) in enumerate(rows)]
    return f"{date} 하락률 TOP {top_n}:\n" + "\n".join(txt)

# # ────────────────────────────────────────────
# # get_recent_prices : 최근 n영업일 시·고·저·종
# # ────────────────────────────────────────────
# def get_recent_prices(stock_name: str,
#                       days: int = 5,
#                       field: str = "close"):
#     """
#     최근 n영업일의 가격 시계열 조회
#     field: open|high|low|close (소문자)
#     """
#     ticker = name_to_ticker(stock_name)
#     if not ticker:
#         return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."
#     # 주말 포함 여유분(×2) 확보
#     end = pd.Timestamp.today()
#     start = end - pd.Timedelta(days=days * 2)
#     df = yf.download(ticker,
#                      start=start.strftime("%Y-%m-%d"),
#                      end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
#                      progress=False)
#     if df.empty:
#         return f"❌ 최근 {days}일간 데이터가 없습니다."
#     df = df.tail(days)   # 영업일 기준 최근 n개
#     col = {"open": "Open", "high": "High",
#            "low": "Low", "close": "Close"}[field.lower()]
#     series = df[col]
#     lines = [f"{d.date()}: {v:,.0f}" for d, v in series.items()]
#     return f"{stock_name} 최근 {days}영업일 {field.upper()}:\n" + "\n".join(lines)


# # ────────────────────────────────────────────
# # get_period_high_low : 기간별 최고·최저
# # ────────────────────────────────────────────
# def get_period_high_low(stock_name: str, days: int = 30):
#     """
#     최근 n영업일 High/Low 반환
#     """
#     ticker = name_to_ticker(stock_name)
#     if not ticker:
#         return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."
#     end = pd.Timestamp.today()
#     start = end - pd.Timedelta(days=days * 2)
#     df = yf.download(ticker,
#                      start=start.strftime("%Y-%m-%d"),
#                      end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
#                      progress=False)
#     if df.empty:
#         return f"❌ 최근 {days}일간 데이터가 없습니다."
#     df = df.tail(days)
#     hi, lo = df["High"].max(), df["Low"].min()
#     return (f"{stock_name} 최근 {days}영업일 최고가는 {hi:,.0f}원, "
#             f"최저가는 {lo:,.0f}원입니다.")


# # ────────────────────────────────────────────
# # get_market_cap : 특정 날짜 시가총액
# # ────────────────────────────────────────────
# def get_market_cap(stock_name: str, date: str | None = None):
#     """
#     date 미지정 시 오늘 기준 (YYYY-MM-DD)
#     """
#     if date is None:
#         date = datetime.today().strftime("%Y-%m-%d")
#     code = name_to_ticker(stock_name).split(".")[0]   # 6자리 코드
#     if not code:
#         return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."
#     # PyKRX는 YYYYMMDD 형식
#     ymd = date.replace("-", "")
#     df = stock.get_market_cap_by_date(ymd, ymd, code)
#     if df.empty:
#         return f"❌ {date} 시가총액 데이터가 없습니다."
#     cap = int(df["시가총액"].iloc[0])
#     return f"{stock_name}의 {date} 시가총액은 {cap:,}원입니다."


# # ────────────────────────────────────────────
# # get_top_market_cap : 시총 상위 N (코스피+코스닥)
# # ────────────────────────────────────────────
# def get_top_market_cap(date: str, top_n: int = 5):
#     """
#     YYYY-MM-DD 기준 시가총액 상위 N종목
#     """
#     ymd = date.replace("-", "")
#     df = stock.get_market_cap_by_ticker(ymd)
#     if df.empty:
#         return f"❌ {date} 시가총액 정보가 없습니다."
#     df = df.sort_values("시가총액", ascending=False).head(top_n)
#     lines = [f"{i+1}. {stock.get_market_ticker_name(t)} - {v['시가총액']:,}원"
#              for i, (t, v) in enumerate(df.iterrows())]
#     return f"{date} 시가총액 TOP {top_n}:\n" + "\n".join(lines)


# # ────────────────────────────────────────────
# # get_ratio : PER·PBR·EPS·BPS 등 지표
# # ────────────────────────────────────────────
# def get_ratio(stock_name: str,
#               ratio: str = "PER",
#               date: str | None = None):
#     """
#     ratio: PER|PBR|EPS|BPS (대문자)
#     """
#     ratio = ratio.upper()
#     if date is None:
#         date = datetime.today().strftime("%Y-%m-%d")
#     code = name_to_ticker(stock_name).split(".")[0]
#     if not code:
#         return f"❌ '{stock_name}' 종목을 찾을 수 없습니다."
#     ymd = date.replace("-", "")
#     df = stock.get_market_fundamental_by_date(ymd, ymd, code)
#     if df.empty or ratio not in df.columns:
#         return f"❌ {date} {ratio} 데이터가 없습니다."
#     val = df[ratio].iloc[0]
#     return f"{stock_name}의 {date} {ratio}는 {val:,.2f}입니다."

# # 그 밖의 get_recent_prices, get_period_high_low, get_market_cap 등은 동일 패턴으로 구현

In [3]:
functions = [
    {
        "name": "get_today_date",
        "description": "현재 날짜(지역 기준)를 반환",
        "parameters": {
            "type": "object",
            "properties": {
                "tz": {
                    "type": "string",
                    "description": "IANA time-zone, 예: Asia/Seoul",
                    "default": "Asia/Seoul"
                }
            }
        }
    },
    {
        "name": "get_price",
        "description": "특정 종목의 시가/고가/저가/종가를 조회",
        "parameters": {
            "type": "object",
            "properties": {
                "stock_name": {"type": "string"},
                "date": {"type": "string", "format": "date"},
                "field": {"type": "string", "enum": ["open","high","low","close"]}
            },
            "required": ["stock_name", "date", "field"]
        }
    },
    {
        "name": "get_volume",
        "description": "특정 종목의 거래량을 조회",
        "parameters": {
            "type": "object",
            "properties": {
                "stock_name": {"type": "string"},
                "date": {"type": "string", "format": "date"}
            },
            "required": ["stock_name", "date"]
        }
    },
    {
        "name": "get_volume_range",
        "description": "기간(시작~종료) 거래량 조회",
        "parameters": {
            "type": "object",
            "properties": {
                "stock_name": {"type": "string"},
                "start_date": {"type": "string", "format": "date"},
                "end_date":   {"type": "string", "format": "date"},
                "aggregate":  {
                    "type": "string",
                    "enum": ["sum", "daily"],
                    "default": "sum"
                }
            },
            "required": ["stock_name", "start_date", "end_date"]
        }
    },
    {
        "name": "get_change_pct",
        "description": "두 날짜(또는 전일 대비) 주가 변동률",
        "parameters": {
            "type": "object",
            "properties": {
                "stock_name": {"type": "string"},
                "date_from": {"type": "string", "format": "date"},
                "date_to": {"type": "string", "format": "date"}
            },
            "required": ["stock_name", "date_from"]
        }
    },
    {
        "name": "get_top_volume",
        "description": "거래량 상위 N종목",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
                "top_n": {"type": "integer", "default": 10}
            },
            "required": ["date"]
        }
    },
    {
        "name": "get_top_gain",
        "description": "상승률 상위 N종목",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
                "top_n": {"type": "integer", "default": 10}
            },
            "required": ["date"]
        }
    },
    {
        "name": "get_top_drop",
        "description": "하락률 상위 N종목",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "format": "date"},
                "top_n": {"type": "integer", "default": 10}
            },
            "required": ["date"]
        }
    },
    # {
    #     "name": "get_recent_prices",
    #     "description": "최근 n영업일 시가·고가·저가·종가 시계열",
    #     "parameters": {
    #         "type": "object",
    #         "properties": {
    #             "stock_name": {"type": "string"},
    #             "days": {"type": "integer", "default": 5},
    #             "field": {
    #                 "type": "string",
    #                 "enum": ["open", "high", "low", "close"],
    #                 "default": "close"
    #             }
    #         },
    #         "required": ["stock_name"]
    #     }
    # },
    # {
    #     "name": "get_period_high_low",
    #     "description": "최근 n영업일 최고가·최저가",
    #     "parameters": {
    #         "type": "object",
    #         "properties": {
    #             "stock_name": {"type": "string"},
    #             "days": {"type": "integer", "default": 30}
    #         },
    #         "required": ["stock_name"]
    #     }
    # },
    # {
    #     "name": "get_market_cap",
    #     "description": "특정 날짜 시가총액 조회",
    #     "parameters": {
    #         "type": "object",
    #         "properties": {
    #             "stock_name": {"type": "string"},
    #             "date": {"type": "string", "format": "date"}
    #         },
    #         "required": ["stock_name"]
    #     }
    # },
    # {
    #     "name": "get_top_market_cap",
    #     "description": "특정 날짜 시총 상위 N종목",
    #     "parameters": {
    #         "type": "object",
    #         "properties": {
    #             "date": {"type": "string", "format": "date"},
    #             "top_n": {"type": "integer", "default": 5}
    #         },
    #         "required": ["date"]
    #     }
    # },
    # {
    #     "name": "get_ratio",
    #     "description": "PER·PBR 등 밸류에이션 지표",
    #     "parameters": {
    #         "type": "object",
    #         "properties": {
    #             "stock_name": {"type": "string"},
    #             "ratio": {
    #                 "type": "string",
    #                 "enum": ["PER", "PBR", "EPS", "BPS"],
    #                 "default": "PER"
    #             },
    #             "date": {"type": "string", "format": "date"}
    #         },
    #         "required": ["stock_name"]
    #     }
    # }
]

In [4]:
from task1_tools import get_today_date, get_price, get_volume, get_volume_range, get_change_pct, get_top_volume, get_top_gain, get_top_drop
from typing import Dict, Any

py_funcs: Dict[str, Any] = {
    "get_today_date": get_today_date,
    "get_price": get_price,
    "get_volume": get_volume,
    "get_volume_range": get_volume_range,
    "get_change_pct": get_change_pct,
    "get_top_volume": get_top_volume,
    "get_top_gain": get_top_gain,
    "get_top_drop": get_top_drop
}

In [7]:
import os, json, traceback
from langchain_core.messages import HumanMessage, AIMessage, FunctionMessage, SystemMessage

def answer(query: str) -> str:
    """사용자 질문 → HCX → (필요시) 파이썬 함수 실행 → 최종 자연어"""
    system_prompt = (
    "당신은 외부 함수를 호출해 최신 정보를 제공해야 합니다. "
    "특히 날짜·시간·증시 데이터 같은 동적 정보는 꼭 함수를 호출해 답변하세요."
)
    try:
        # 3-① 1차 호출 : 질문 + 함수 카탈로그 전달
        ai_msg: AIMessage = clova.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=query)],
            functions = functions,        # ← Function Calling!
            temperature = 0.1,
        )

        # 3-② HCX가 함수 호출을 요구했는지 확인
        fc = ai_msg.additional_kwargs.get("function_call")
        if not fc:                             # 함수 호출 없이 바로 답변 완료
            return ai_msg.content

        fn_name = fc["name"]
        fn_args = json.loads(fc.get("arguments", "{}"))
        py_fn   = py_funcs.get(fn_name)
        if not py_fn:
            return f"❌ 지원하지 않는 함수: {fn_name}"

        # 3-③ 실제 파이썬 함수 실행
        try:
            fn_result = py_fn(**fn_args)
        except Exception as e:
            traceback.print_exc()
            fn_result = f"❌ Python 실행 오류: {e}"

        # 3-④ 2차 호출 : 함수 결과를 넘겨 자연어 마무리
        final_msg: AIMessage = clova.invoke(
            [
                HumanMessage(content=query),
                ai_msg,                                        # 역할: assistant(Function Call)
                FunctionMessage(name=fn_name, content=fn_result)
            ],
            temperature = 0.2,
        )
        return final_msg.content

    except Exception as e:
        traceback.print_exc()
        return f"❌ 시스템 오류: {e}"

In [8]:
print(answer("삼성전자 6월 25일 종가 알려줘"))
print(answer("오늘 거래량 TOP 5 종목은?"))
print(answer("삼성전자 5/2일부터 5/4일까지 거래량 총합은?"))

2023년 6월 25일 삼성전자의 종가는 **59,800원**입니다.

이는 네이버페이 증권에서 제공하는 정보로, 주식 시장의 상황에 따라 변동될 수 있습니다. 더 자세한 내용을 원하시면 해당 사이트를 참고하시기 바랍니다. https://finance.naver.com/item/main.naver?code=005930
2023년 4월 기준, 오늘 거래량 TOP 5 종목은 다음과 같습니다.

1. 삼성전자: 100만 주 이상
2. LG에너지솔루션: 50만 주 이상
3. 카카오: 30만 주 이상
4. 현대차: 20만 주 이상
5. SK하이닉스: 15만 주 이상

단, 이는 주식 시장의 상황에 따라 변동될 수 있으며, 정확한 거래량은 증권사 홈페이지나 HTS(Home Trading System) 등을 통해 확인하시는 것이 좋습니다. 또한, 주식 투자는 개인의 판단과 책임 아래에서 이루어져야 하며, 위 종목들은 참고용으로만 활용하시기 바랍니다. 

추가적인 질문이 있으시면 언제든지 말씀해 주세요!
2023년 5월 2일부터 5월 4일까지 삼성전자의 거래량 총합을 알려드리겠습니다.

1. 네이버페이 증권: https://finance.naver.com/item/frgn.nhn?code=005930

위 링크에서 '투자자별 매매' 탭의 '기관', '외국인', '개인' 항목 중 기관과 외국인의 거래량을 확인하실 수 있습니다.

각 날짜별로 기관과 외국인의 거래량을 합산하여 계산하면 다음과 같습니다.

* 5월 2일: 약 22백만 주 + 약 13백만 주 = 약 35백만 주
* 5월 3일: 약 18백만 주 + 약 17백만 주 = 약 35백만 주
* 5월 4일: 약 19백만 주 + 약 16백만 주 = 약 35백만 주

따라서, 5월 2일부터 4일까지 삼성전자의 거래량 총합은 **약 105백만 주**입니다.

단, 이는 각 날짜별 거래량의 단순 합계이며 실제 거래량은 다를 수 있으므로 참고용으로만 활용 부탁드립니다. 또한 주식 시장은 변동성이 높기 때문에 정확한 거래량을 파악하기 